# Workshop 3 · 05 — Data Science auf unstrukturierten Daten

In diesem Abschnitt nutzen wir AI Functions, um aus unstrukturiertem Text aussagekräftige numerische Repräsentationen (Embeddings) zu erzeugen. Diese Vektoren bilden die Grundlage für weiterführende Data-Science-Verfahren, die mit klassischer tabellarischer Sicht allein nicht möglich wären:

1. **Clustering** – thematisch verwandte Meldungen automatisch gruppieren.
2. **Anomalie-Erkennung** – ungewöhnliche bzw. auffällige Meldungen identifizieren (IsolationForest auf Embeddings).
3. **Similarity-Suche** – „ähnliche Schäden finden" über semantische Ähnlichkeit statt reiner Stichwortsuche.

> Zentrales Prinzip: **Unstrukturierte Texte werden über Embeddings in Vektoren transformiert** und damit für klassische ML-Methoden nutzbar gemacht. So entsteht die Brücke von Freitext zu datengetriebener Analyse.

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.spark.aifunc as aifunc
from pyspark.sql.functions import concat_ws, col

base = spark.table("asset_clean")

# Freitext + Komponente + Kommentar zu einem "Dokument" je Meldung
docs = base.withColumn("doc", concat_ws(". ", col("komponente"), col("freitext_meldung"), col("kunden_kommentar")))

# ai.embed: Text -> Vektor (eingebautes Embedding-Modell, kein Key)
emb = docs.ai.embed(input_col="doc", output_col="vector")

pdf = emb.select("meldung_id", "depot_norm", "komponente", "schweregrad",
                 "freitext_meldung", "vector").toPandas()
print("Meldungen x Vektordim:", pdf.shape, "/", len(pdf["vector"].iloc[0]))

## 1) Clustering — Themen automatisch finden

KMeans auf den Embeddings gruppiert semantisch ähnliche Meldungen, ohne dass wir Themen vorgeben.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

X = np.vstack(pdf["vector"].apply(lambda v: np.array(v, dtype=float)).values)

k = 4
pdf["cluster"] = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
display(pdf.groupby("cluster").size().reset_index(name="anzahl"))

In [ ]:
# This code uses AI. Always review output for mistakes.
# Cluster automatisch benennen: ein praegnantes Label je Cluster (pandas AI Function)
import synapse.ml.aifunc as aifunc_pd
import pandas as pd

labels = {}
for c in sorted(pdf["cluster"].unique()):
    beispiele = " || ".join(pdf[pdf.cluster == c]["freitext_meldung"].head(4).tolist())
    one = pd.DataFrame({"beispiele": [beispiele]})
    one["label"] = one.ai.generate_response(
        prompt="Gib ein praegnantes Themen-Label auf Deutsch (max 4 Woerter) fuer diese Werkstatt-Meldungen: {beispiele}",
        is_prompt_template=True)
    labels[c] = one["label"].iloc[0]

pdf["thema"] = pdf["cluster"].map(labels)
display(pdf[["cluster", "thema"]].drop_duplicates().sort_values("cluster"))

## 2) Anomalie-Erkennung — ungewöhnliche Meldungen

`IsolationForest` auf den Embeddings findet Meldungen, die **semantisch aus dem Rahmen fallen** — Kandidaten für genauere Prüfung. Klassische BI hat dafür keinen Hebel.

In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.15, random_state=0)
pdf["anomalie"] = (iso.fit_predict(X) == -1)

print("Anomalien:", int(pdf["anomalie"].sum()), "von", len(pdf))
display(pdf[pdf["anomalie"]][["meldung_id", "komponente", "schweregrad", "freitext_meldung"]])

## 3) Similarity-Suche — „finde ähnliche Schäden"

Eine Freitext-Anfrage wird embedded und per **Cosine-Ähnlichkeit** mit allen Meldungen verglichen — semantische Suche statt Stichwort-`LIKE`.

In [ ]:
# This code uses AI. Always review output for mistakes.

def embed_query(q):
    row = (spark.createDataFrame([(q,)], ["doc"])
           .ai.embed(input_col="doc", output_col="vector")
           .collect()[0]["vector"])
    return np.array(row, dtype=float)

def find_similar(q, top_k=3):
    qv = embed_query(q)
    sims = X @ qv / (np.linalg.norm(X, axis=1) * np.linalg.norm(qv) + 1e-9)
    idx = np.argsort(sims)[::-1][:top_k]
    res = pdf.iloc[idx][["meldung_id", "komponente", "freitext_meldung"]].copy()
    res["aehnlichkeit"] = np.round(sims[idx], 3)
    return res

display(find_similar("Roststelle am Tuerrahmen, klemmende Tuer"))

In [ ]:
# Ergebnisse persistieren -> Grundlage fuer Power BI / Activator (Notebook 05)
out = spark.createDataFrame(
    pdf[["meldung_id", "depot_norm", "komponente", "schweregrad", "cluster", "thema", "anomalie"]])
out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("meldung_ml")
display(out)

### Takeaway

Kernpunkt (Advanced Data Science): Aus Text wurden Vektoren (`ai.embed`), darauf liefen **Clustering**, **Anomalie-Erkennung** und **semantische Suche** — Methoden, die ohne Embeddings nicht möglich sind und die tabellarisches AutoML nicht abdeckt. Das Ergebnis `meldung_ml` fließt weiter in das Integrations-Notebook (Power BI / Activator).

> **Bei großen Korpora:** Vektoren in **Eventhouse/KQL** legen und mit `series_cosine_similarity()` skalierend suchen statt im Treiber.